# Set Up

In [1]:
#Import libraries
import pandas as pd
import numpy as np
import re, os, sqlite3
from datetime import datetime

### Read in Data

In [106]:
path = '/Users/minji.kang/Documents/NGDT/Data_export_management/PMHSA/english_eleven/'

def read_and_bind_df(mypath) :
    all_files = os.listdir(mypath)
    report_files = [file for file in all_files if file.startswith('report')]
    report_df = []
    for i in range(len(report_files)):
        temp_df = pd.read_csv(os.path.join(mypath, report_files[i]), encoding='ISO-8859-1')
        report_df.append(temp_df)
    report = pd.concat(report_df, ignore_index=True)
    report.rename(columns={report.columns[0]: 'id'}, inplace=True) 
    return report

df = read_and_bind_df(path)
# df.to_csv(os.path.join('/Users/minji.kang/','report_all.csv'),index=False)

In [107]:
# df.to_csv(os.path.join('/Users/minji.kang/','report_all.csv'),index=False)

### Add Timezone offset

In [108]:
def add_timezone_offset(mydata, columntoaddto):
    offset_added = []
    for i in range(len(mydata[columntoaddto])):
        if mydata['timezone_offset'][i] != 'nan':
            temp = pd.to_numeric(mydata[columntoaddto][i], errors='coerce') + (pd.to_numeric(mydata['timezone_offset'][i], errors='coerce')* 60 * 1000)
        else: 
            temp = pd.to_numeric(mydata[columntoaddto][i], errors='coerce')
        offset_added.append(temp)
    return offset_added

df['start_Time'] = add_timezone_offset(df, 'activity_start_time')
df['end_Time'] = add_timezone_offset(df, 'activity_end_time')  
df['schedule_Time'] = add_timezone_offset(df, 'activity_scheduled_time')  

### Group by min start_Time & max End_Time

In [109]:
dat_processed = df.groupby(['secret_user_id', 'activity_flow_id', 'activity_scheduled_time'], group_keys=True).apply(lambda x: x.assign(start_Time=x['start_Time'].min(), end_Time=x['end_Time'].max())).reset_index(drop=True)

In [110]:
def val_score_mapping(data):
    response_scores = []  # List to store results

    for i in range(len(data['response'])):
        options = data['options'][i]
        response = data['response'][i]
        clean_response = re.sub(r"value: |geo: ", "", response)

        # Ensure 'options' and 'response' are valid strings
        if not isinstance(options, str) or not isinstance(response, str):
            response_scores.append(clean_response)  # Append NaN for invalid rows
            continue

        if re.search(r'score: ', options):
            split_options = options.strip().split("),")
            split_response = response.strip().split(": ")[1].split(',')
            scores = {}

            for j in split_options:
                if "(score" in j:  # Ensure the string contains the expected structure
                    val_parts = j.split("(score")
                    if len(val_parts) == 2 and ": " in val_parts[0]:
                        val_num = val_parts[0].split(": ")[1].strip()
                        score_num = val_parts[1].split(": ")[1].strip(" )")
                        scores[val_num] = score_num

            response_score_mapping = {
                resp.strip(): scores.get(resp.strip(), "N/A")  # Handle missing mappings
                for resp in split_response
            }
            response_scores.append(', '.join(response_score_mapping.values()))
        else:
            response_scores.append(clean_response)  # Append NaN if no valid scores are found

    return pd.Series(response_scores)

In [111]:
dat_processed['response_scores'] = val_score_mapping(dat_processed)